# 실험 B 캐시 - 슬라이스 분산 배치

224px, 150mm, 6슬롯

시리즈마다 인접 3장 묶음을 슬라이스 범위의 25% / 50% / 75% 에 하나씩 둔다

GPU 를 쓰지 않는다


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

T0 = time.time()
for _c in ['/kaggle/input/datasets/hyejinkwak/knee-src', '/kaggle/input/knee-src']:
    if os.path.isdir(_c):
        sys.path.insert(0, _c); print('src:', _c); break
else:
    raise FileNotFoundError('knee-src 없음')

import paths
wheels = paths.find_input('knee-dicom-wheels')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                '--find-links', str(wheels),
                'pylibjpeg', 'pylibjpeg-libjpeg', 'pylibjpeg-openjpeg', 'python-gdcm'],
               check=True)
import pydicom
import config, dicom_io, cache
print('pydicom', pydicom.__version__)
print(paths.describe())
config.verify_labels()
print(f'준비 {time.time()-T0:.1f}s')


In [ ]:
import numpy as np, pandas as pd

OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
# 실험 B - 인접 3장 묶음을 슬라이스 범위의 25% / 50% / 75% 에 하나씩, 크롭 150mm
# 기존 캐시는 중앙 연속 9장(약 30mm)만 봐서 내외측 구획을 놓칠 수 있다 (journey.md 9장)
cfg = cache.CacheConfig(out_px=224, n_slices=9, extent_mm=150.0, layout='spread',
                        triplet_centers=(0.25, 0.5, 0.75))
cfg.slots = dicom_io.SLOTS[:6]   # plan C 가 실제로 쓰는 6칸
SHARD = 500

train = pd.read_csv(paths.TRAIN_CSV)
studies = train.StudyInstanceUID.tolist()
n_shards = (len(studies) + SHARD - 1) // SHARD

print(f'{len(studies):,} studies -> {n_shards} shards x {SHARD}')
print('슬롯:', cfg.slots)
print('예상 용량:', cache.estimate_size(len(studies), cfg))
one = SHARD * len(cfg.slots) * cfg.n_slices * cfg.out_px**2 / 1024**3
print(f'샤드 하나 {one:.2f} GiB')

# 실행 시점 검사 - 틀린 설정으로 2시간을 돌지 않게 한다
# knee-src 가 옛날 버전이면 슬롯 수가 다르다 (실제로 8칸 대신 6칸으로 빌드한 적 있음)
EXPECT_SLOTS, EXPECT_PX = 6, 224
assert len(cfg.slots) == EXPECT_SLOTS, (
    f'슬롯이 {len(cfg.slots)}개다. {EXPECT_SLOTS}개여야 한다. '
    f'knee-src 가 최신인지 확인할 것: bash scripts/push_src.sh')
assert cfg.out_px == EXPECT_PX, f'out_px {cfg.out_px} != {EXPECT_PX}'
assert cfg.n_slices == 9, 'n_slices 가 9가 아니다'
assert cfg.layout == 'spread' and cfg.extent_mm == 150.0, '실험 B 설정이 아니다'
assert cache.slice_indices(30, cfg) == [6, 7, 8, 13, 14, 15, 21, 22, 23], \
    'knee-src 가 옛날 버전이다: bash scripts/push_src.sh'
assert cfg.out_px % 14 == 0, 'DINOv2 패치 14 의 배수여야 한다'
print(f'설정 검증 통과: 슬롯 {len(cfg.slots)}개, {cfg.out_px}px')


## 빌드

샤드마다 `.npy` 하나와 인덱스 행들을 남긴다
중간에 실패해도 그때까지의 샤드는 출력에 남는다


In [ ]:
rows = []
t_build = time.time()

for si in range(n_shards):
    chunk = studies[si*SHARD : (si+1)*SHARD]
    arr = np.zeros((len(chunk), len(cfg.slots), cfg.n_slices,
                    cfg.out_px, cfg.out_px), np.uint8)
    t0 = time.time()
    n_err = 0
    for j, u in enumerate(chunk):
        a, info = cache.build_study(u, cfg)
        arr[j] = a
        n_err += len(info.errors)
        r = {'study': u, 'shard': si, 'row': j, 'n_filled': info.n_filled,
             'n_err': len(info.errors)}
        for s in cfg.slots:
            r[s] = int(s in info.slots_filled)
            r[f'{s}__mm'] = round(info.mm_per_px.get(s, 0.0), 4)
        rows.append(r)

    fp = OUT / f'cache_{si:02d}.npy'
    np.save(fp, arr)
    del arr

    done = (si + 1) * SHARD
    el = time.time() - t_build
    eta = el / min(done, len(studies)) * (len(studies) - min(done, len(studies)))
    print(f'shard {si:02d}  {len(chunk):>4} studies  {time.time()-t0:6.0f}s  '
          f'err {n_err:>3}  {fp.stat().st_size/1024**3:.2f} GiB  '
          f'경과 {el/60:.0f}분  남은 예상 {eta/60:.0f}분', flush=True)

print(f'\n빌드 완료 {(time.time()-t_build)/60:.1f}분')


## 인덱스와 요약


In [ ]:
idx = pd.DataFrame(rows)
idx.to_csv(OUT / 'cache_index.csv', index=False)
print(f'index {idx.shape}')
print()
print('=== 슬롯별 채움률 ===')
print(idx[list(cfg.slots)].mean().round(3).to_string())
print()
print('=== 스터디당 채운 슬롯 수 ===')
print(idx.n_filled.value_counts().sort_index().to_string())
print()
print(f'에러가 있는 스터디: {(idx.n_err > 0).sum()}')
print(f'슬롯이 하나도 안 찬 스터디: {(idx.n_filled == 0).sum()}')


In [ ]:
meta = {
    'n_studies': int(len(idx)), 'shard_size': SHARD, 'n_shards': int(n_shards),
    'slots': list(cfg.slots), 'n_slices': cfg.n_slices, 'out_px': cfg.out_px,
    'extent_mm': cfg.extent_mm, 'mm_per_px': round(cfg.extent_mm/cfg.out_px, 4),
    'layout': cfg.layout, 'triplet_centers': list(cfg.triplet_centers),
    'dtype': 'uint8',
    'shape_per_shard': [SHARD, len(cfg.slots), cfg.n_slices, cfg.out_px, cfg.out_px],
    'slot_fill_rate': idx[list(cfg.slots)].mean().round(4).to_dict(),
    'build_minutes': round((time.time()-T0)/60, 1),
}
(OUT / 'cache_meta.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print(json.dumps(meta, indent=2, ensure_ascii=False))

files = sorted(os.listdir(OUT))
total = sum((OUT/f).stat().st_size for f in files) / 1024**3
print(f'\n출력 {len(files)}개  {total:.2f} GiB  (한도 20GB)')
print(files)


## 확인용 이미지

샤드에서 한 건 꺼내 제대로 저장됐는지 본다


In [ ]:
import matplotlib.pyplot as plt

a = np.load(OUT / 'cache_00.npy', mmap_mode='r')[0]
fig, axes = plt.subplots(len(cfg.slots), cfg.n_slices,
                         figsize=(cfg.n_slices*1.25, len(cfg.slots)*1.4))
for i, slot in enumerate(cfg.slots):
    for j in range(cfg.n_slices):
        ax = axes[i, j]
        ax.imshow(a[i, j], cmap='gray', vmin=0, vmax=255)
        ax.set_xticks([]); ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(slot, fontsize=7,
                          color='black' if idx.iloc[0][slot] else 'red')
fig.suptitle(f'cache_00.npy row 0   {idx.iloc[0].n_filled}/{len(cfg.slots)} slots', fontsize=10)
plt.tight_layout()
fig.savefig(OUT / 'cache_check.png', dpi=95, bbox_inches='tight')
plt.show()
